In [7]:
from eoflow.rivers import get_river_network_from_poly 
from shapely.geometry import Point, box
import geopandas as gpd
import folium

In [8]:
query = """
[out:json];
node(50.7,-3.6,50.8,-3.5);
out 1;
"""

center = Point((50.7260, -3.52756))

lat_range, long_range = 0.2, 0.4

quad = box(
    center.y - long_range, center.x - lat_range,
    center.y + long_range, center.x + lat_range
)

osm_rivers = get_river_network_from_poly(
    quad,
    use_bbox=True,
)

2026-06-24 16:36:16,392 - eoflow.rivers - DEBUG - Built Overpass bbox query for bounds: (-3.9275599999999997, 50.525999999999996, -3.12756, 50.926)
2026-06-24 16:36:16,394 - eoflow.rivers - INFO - Querying Overpass for waterways in bbox: (-3.9275599999999997, 50.525999999999996, -3.12756, 50.926)
2026-06-24 16:36:16,395 - eoflow.rivers - DEBUG - Overpass request attempt=1/6 endpoint=https://overpass-api.de/api/interpreter
2026-06-24 16:36:16,557 - eoflow.rivers - ERROR - Unexpected requests error: 406 Client Error: Not Acceptable for url: https://overpass-api.de/api/interpreter
Traceback (most recent call last):
  File "/home/finley/Work/RDS/projects/enforce/repos/eoflow/src/eoflow/rivers.py", line 247, in _fetch_overpass
    resp.raise_for_status()
  File "/home/finley/Work/RDS/projects/enforce/repos/eoflow/.venv/lib/python3.12/site-packages/requests/models.py", line 1167, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 406 Client 

ValueError: Unknown column geometry

In [ ]:


# import requests

# resp = requests.post(
#     "https://lz4.overpass-api.de/api/interpreter",
#     data={"data": query},
#     headers={"User-Agent": "test-overpass"},
#     timeout=30,
# )

# print(resp.status_code)
# print(resp.text[:500])

In [ ]:
osm_rivers = get_river_network_from_shape(quad)

In [ ]:

# Get the center of your area
m = folium.Map(
    location=[center.x, center.y], 
    zoom_start=10,
    scroll_wheel_zoom=False, 
    height="500px", 
    width="800px",
)

In [ ]:
area = gpd.GeoDataFrame(geometry=[quad], crs=osm_rivers.crs)

folium.GeoJson(
    area,
    name="Bounding Box",
    style_function=lambda x: {"color": "red", "weight": 2, "fill": False}
).add_to(m)

In [ ]:
masked_rivers = gpd.overlay(osm_rivers, area, how="intersection")

In [ ]:
# Ensure your GeoDataFrame is in WGS84 (lat/lon)
masked_rivers = masked_rivers.to_crs("EPSG:4326")

# Add all masked rivers as a GeoJson layer
folium.GeoJson(
    masked_rivers,
    name="Rivers",
    style_function=lambda x: {"color": "blue", "weight": 2}
).add_to(m)


In [ ]:
m

In [ ]:
masked_rivers

In [ ]:
type(masked_rivers)

In [ ]:
masked_rivers.to_file("osm_rivers.gdf")